In [2]:
import os
import sys
import time

import numpy as np
import scipy

In [3]:
sys.path.insert(0, '../OptimalNumberOfTopics')

In [4]:
import topnum

from topnum.scores.perplexity_score import PerplexityScore
from topnum.scores.diversity_score import DiversityScore, KNOWN_METRICS
from topnum.model_constructor import init_model_from_family, KnownModel, PARAMS_EXPLORED

from topnum.regularizers import (
    FastFixPhiRegularizer, DecorrelateWithOtherPhiRegularizer, DecorrelateWithOtherPhiRegularizer2
)
from topnum.scores.intratext_coherence_score import (
    IntratextCoherenceScore,
    ComputationMethod,
    WordTopicRelatednessType,
)

In [5]:
import topicnet
from topicnet.cooking_machine.dataset import Dataset
from topicnet.cooking_machine.models import (
    BaseScore as BaseTopicNetScore,
    TopicModel
)
from topicnet.cooking_machine.models.thetaless_regularizer import (
    dataset2sparse_matrix,
)

In [6]:
topicnet.__file__

! ls /home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/dataset_manager/

20NG.csv	 MKB10__internals	    RTL_Wiki.csv
20NG__internals  postnauka.csv		    RTL_Wiki_person.csv
api.py		 postnauka__internals	    RTL_Wiki_person__internals
Brown		 postnauka_noow.csv	    ruwiki_good__internals
Brown_BOW.csv	 postnauka_noow__internals  ruwiki_good.txt
Brown_NOOW.csv	 __pycache__		    WikiRef-220
hf		 Reuters		    wiki_ref220_bow.csv
__init__.py	 Reuters_BOW.csv	    wiki_ref220_natural_order.csv
MKB10.csv	 Reuters_NOOW.csv


In [7]:
DATA_FOLDER_PATH = '/data_mil/shared/CompressaAI/iterative/data/noow'

In [8]:
! ls $DATA_FOLDER_PATH

_20_Newsgroups.csv	       MKB_10_NOOW__internals
_20_Newsgroups__internals      Post_Science__internals
20_Newsgroups_NOOW.csv	       Post_Science_NOOW.csv
20_Newsgroups_NOOW__internals  Post_Science_NOOW_fixed.csv
_Lenta.csv		       Post_Science_NOOW_fixed__internals
MKB_10__internals	       Post_Science_NOOW__internals
MKB_10_NOOW.csv		       WikiRef_220_NOOW.csv


In [9]:
dataset = Dataset(
    f'{DATA_FOLDER_PATH}/MKB_10_NOOW.csv',
)

dataset.get_possible_modalities()

{'@letter', '@ngram', '@text'}

In [10]:
dataset._internals_folder_path

'/data_mil/shared/CompressaAI/iterative/data/noow/MKB_10_NOOW__internals'

In [11]:
MAIN_MODALITY = '@text'

In [12]:
dataset._data.head()

,id,raw_text,vw_text
id,,,
«Бедная_симптомами»_шизофрения,«Бедная_симптомами»_шизофрения,«Бе́дная симпто́мами» шизофрени́я — подтип шиз...,«Бедная_симптомами»_шизофрения |@text бедный с...
"46,XX/46,XY","46,XX/46,XY","46,XX/46,XY (тетрагаметный химеризм) — это раз...","46,XX/46,XY |@text <person> химеризм разновидн..."
"Синдром_48,_XXXY","Синдром_48,_XXXY","Синдром 48, XXXY — это генетическое состояние,...","Синдром_48,_XXXY |@text синдром xxxy генетичес..."
"Синдром_48,_XXYY","Синдром_48,_XXYY","Синдром 48, XXYY — это аномалия хромосом, при ...","Синдром_48,_XXYY |@text синдром xxyy аномалия ..."
"Синдром_48,_XYYY","Синдром_48,_XYYY","Синдром 48, XYYY — чрезвычайно редкая анеуплои...","Синдром_48,_XYYY |@text синдром xyyy чрезвычаи..."


In [13]:
dataset._data.shape

(2036, 3)

In [14]:
dataset.get_dictionary()

artm.Dictionary(name=849b3a43-4e71-459c-90df-177ca1afa29f, num_entries=244551)

In [15]:
dictionary = dataset.get_dictionary()

In [16]:
print(dictionary)

for modality in dataset.get_possible_modalities():
    if modality not in [MAIN_MODALITY]:
        dictionary.filter(class_id=modality, max_df=0, inplace=True)

artm.Dictionary(name=849b3a43-4e71-459c-90df-177ca1afa29f, num_entries=244551)


In [17]:
dictionary.filter(min_df=2, max_df_rate=0.5)

artm.Dictionary(name=849b3a43-4e71-459c-90df-177ca1afa29f, num_entries=22608)

In [17]:
dictionary.filter(min_df_rate=0.5)  # (min_df=2, max_df_rate=0.5)

artm.Dictionary(name=89f935ad-dfee-4c4e-991e-cddade8a1861, num_entries=0)

In [18]:
dictionary

artm.Dictionary(name=89f935ad-dfee-4c4e-991e-cddade8a1861, num_entries=0)

In [35]:
dictionary.save_text('test_dict.txt')

In [37]:
! cat test_dict.txt

name: 3ed7db55-fc3e-4f86-b0ad-c4ffdcf82fd0 num_items: 2036
token, class_id, token_value, token_tf, token_df
год, @text, 0.00550703052431345, 5426.0, 1094.0
лечение, @text, 0.005373059306293726, 5294.0, 1242.0
что, @text, 0.006797011010348797, 6697.0, 1244.0
как, @text, 0.0069807143881917, 6878.0, 1465.0
развитие, @text, 0.0036872541531920433, 3633.0, 1089.0
являться, @text, 0.005028996616601944, 4955.0, 1330.0
случай, @text, 0.005558792036026716, 5477.0, 1335.0
мочь, @text, 0.009810349904000759, 9666.0, 1602.0
<person>, @text, 0.04563852399587631, 44967.0, 1985.0
заболевание, @text, 0.006939101964235306, 6837.0, 1428.0
<person>_<person>, @ngram, 0.005488390568643808, 3171.0, 1053.0
быть, @text, 0.006246916949748993, 6155.0, 1330.0


In [18]:
dataset._cached_dict = dictionary

In [19]:
dataset.get_dictionary()

artm.Dictionary(name=849b3a43-4e71-459c-90df-177ca1afa29f, num_entries=22608)

In [20]:
def calc_doc_occurrences(dataset, modality):
    """
    :param n_dw_matrix: sparse document-word matrix, shape is D x W
    :return: sparse matrix of co-occurrences

    doc_occurrences[w1, w2] = the number of the documents
    where there are w1 and w2
    """
    n_dw_matrix = dataset2sparse_matrix(dataset, modality, modalities_to_use=[modality])
    matrix = (scipy.sparse.csc_matrix(n_dw_matrix) > 0).astype(int)
    co_occurrences = matrix.T * matrix

    return co_occurrences.diagonal(), co_occurrences


def create_pmi_top_function(
    doc_occurrences, doc_co_occurrences,
    documents_number, top_sizes,
    topic_indices,
    co_occurrences_smooth=1.
):
    """
    :param doc_occurrences: array of doc occurrences of words
    :param doc_co_occurrences: sparse matrix of doc co-occurrences of words
    :param documents_number: number of the documents
    :param top_sizes: list of top values to calculate top-pmi for
    :param co_occurrences_smooth: constant to smooth co-occurrences in log
    :return: function which takes phi and theta and returns
    pair of two arrays: pmi-s of the tops and ppmi-s of the tops

    pmi[i] - pmi(top of size top_sizes[i])
    ppmi[i] - ppmi(top of size top_sizes[i])

    pmi(words) = sum_{u in words, v in words, u != v}
    log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    )

    ppmi(words) = sum_{u in words, v in words, u != v}
    max(log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    ), 0)

    """
    def func(phi):
        _T, W = phi.shape
        T = len(topic_indices)

        max_top_size = max(top_sizes)
        topic_pmis, topic_ppmis = dict(), dict()
        pmi, ppmi = np.zeros(max_top_size), np.zeros(max_top_size)
        tops = np.argpartition(phi, -max_top_size, axis=1)[:, -max_top_size:]
        
        for t in topic_indices:
            top = sorted(tops[t], key=lambda w: - phi[t, w])
            co_occurrences = doc_co_occurrences[top, :][:, top].todense()
            occurrences = doc_occurrences[top]
            values = np.log(
                (co_occurrences * documents_number + co_occurrences_smooth)
                / (occurrences[:, np.newaxis] * occurrences[np.newaxis, :] + co_occurrences_smooth)
            )
            diag = np.diag_indices(len(values))
            # values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()

            current_pmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_pmis[t] = current_pmi
            pmi += current_pmi

            values[values < 0.] = 0.
            current_ppmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_ppmis[t] = current_ppmi
            ppmi += current_ppmi
            
        sizes = np.arange(2, max_top_size + 1)
        pmi[1:] /= (T * sizes * (sizes - 1))
        ppmi[1:] /= (T * sizes * (sizes - 1))
        indices = np.array(top_sizes) - 1

        for t in topic_indices:
            topic_pmis[t][1:] /= (sizes * (sizes - 1))
            topic_ppmis[t][1:] /= (sizes * (sizes - 1))

        result_topic_pmis = {t: p[indices] for t, p in topic_pmis.items()}
        result_topic_ppmis = {t: p[indices] for t, p in topic_ppmis.items()}

        return pmi[indices], ppmi[indices], result_topic_pmis, result_topic_ppmis

    return func

In [21]:
%%time

occurences, co_occurences = calc_doc_occurrences(dataset, MAIN_MODALITY)

CPU times: user 3.9 s, sys: 122 ms, total: 4.02 s
Wall time: 3.97 s


In [22]:
co_occurences.shape

(22608, 22608)

In [23]:
dataset.get_dictionary()

artm.Dictionary(name=849b3a43-4e71-459c-90df-177ca1afa29f, num_entries=22608)

In [24]:
calc_pmi = create_pmi_top_function(
    occurences, co_occurences,
    dataset.get_dataset().shape[0], [20],
    topic_indices=[0, 1, 2],
    co_occurrences_smooth=1e-2,
)

In [25]:
class TopTokenCoherence(BaseTopicNetScore):
    def __init__(self, name, func):
        super().__init__()

        self._name = name
        self.calc_pmi = func

    def call(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[1]

    def call_by_topic(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[3]

In [25]:
KnownModel

<enum 'KnownModel'>

In [26]:
PARAMS_EXPLORED

{<KnownModel.LDA: 'LDA'>: {'prior': ['symmetric', 'asymmetric', 'heuristic']},
 <KnownModel.PLSA: 'PLSA'>: {},
 <KnownModel.TLESS: 'TARTM'>: {},
 <KnownModel.SPARSE: 'sparse'>: {'smooth_bcg_tau': [0.05, 0.1],
  'sparse_sp_tau': [-0.05, -0.1]},
 <KnownModel.DECORRELATION: 'decorrelation'>: {'decorrelation_tau': [0.02,
   0.05,
   0.1]},
 <KnownModel.ARTM: 'ARTM'>: {'smooth_bcg_tau': [0.05, 0.1],
  'sparse_sp_tau': [-0.05, -0.1],
  'decorrelation_tau': [0.02, 0.05, 0.1]}}

In [26]:
NUM_TOPICS = 50  # vary
NUM_TRAINS = 3
NUM_ITERATIONS = 20
NUM_TOP_TOKENS = 20

In [27]:
dataset.get_dictionary()

artm.Dictionary(name=849b3a43-4e71-459c-90df-177ca1afa29f, num_entries=22608)

In [28]:
dictionary = dataset.get_dictionary()

## Test

In [45]:
PARAMS_EXPLORED[KnownModel.PLSA]

{}

In [54]:
%%time

model = init_model_from_family(
    family=KnownModel.PLSA,
    dataset=dataset,
    main_modality=MAIN_MODALITY,
    num_topics=NUM_TOPICS,
    seed=1,
)

model._fit(dataset.get_batch_vectorizer(), num_iterations=10)

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


CPU times: user 3min 1s, sys: 2.88 s, total: 3min 4s
Wall time: 1min 16s


In [55]:
list(model.scores.keys())

['PerplexityScore@all',
 'SparsityThetaScore',
 'SparsityPhiScore@lemmatized',
 'PerplexityScore@lemmatized',
 'TopicKernel@lemmatized.average_coherence',
 'TopicKernel@lemmatized.average_contrast',
 'TopicKernel@lemmatized.average_purity',
 'TopicKernel@lemmatized.average_size',
 'TopicKernel@lemmatized.coherence',
 'TopicKernel@lemmatized.contrast',
 'TopicKernel@lemmatized.purity',
 'TopicKernel@lemmatized.size',
 'TopicKernel@lemmatized.tokens']

In [58]:
model.scores[f'PerplexityScore{MAIN_MODALITY}']

[60934.58203125,
 9283.0244140625,
 8148.603515625,
 6405.390625,
 5430.90283203125,
 4980.9423828125,
 4732.5595703125,
 4578.2822265625,
 4477.1103515625,
 4408.48046875]

In [59]:
phi = model.get_phi()
target_topic_indices = list(range(NUM_TOPICS))  # phi.columns.get_loc()
target_topic_names = [phi.columns[i] for i in target_topic_indices]

custom_scores = [
    TopTokenCoherence(
        name=f'coherence_{top}',
        func=create_pmi_top_function(
            occurences, co_occurences,
            dataset.get_dataset().shape[0], [top],
            topic_indices=target_topic_indices,
            co_occurrences_smooth=1e-2,
        )
    )
    for top in [20]  # [10, 20, 50, 100]
]
custom_scores = custom_scores + [
    DiversityScore(
        name=f'diversity_{metric}',
        topic_names=['topic_0', 'topic_1'],
        class_ids=MAIN_MODALITY,
    )

    for metric in KNOWN_METRICS
]

for score in custom_scores:
    res = score.call(model)

    print(score._name)
    print(res)

    if isinstance(score, TopTokenCoherence):
        res_by_topic = score.call_by_topic(model)

        print(res_by_topic)

coherence_20
[0.0273816]
{0: array([0.]), 1: array([0.02958778]), 2: array([0.04763114]), 3: array([0.01888187]), 4: array([0.]), 5: array([0.03611333]), 6: array([0.]), 7: array([0.09164026]), 8: array([0.04763114]), 9: array([0.]), 10: array([0.]), 11: array([0.07011646]), 12: array([0.]), 13: array([0.]), 14: array([0.]), 15: array([0.]), 16: array([0.]), 17: array([0.08207206]), 18: array([0.02869565]), 19: array([0.09526229])}
diversity_euclidean
0.038674582514403096
diversity_jensenshannon
0.038674582514403096
diversity_hellinger
0.038674582514403096
diversity_cosine
0.038674582514403096


In [60]:
model.get_phi(class_ids=MAIN_MODALITY)['topic_18'].sort_values(ascending=False)

modality     token          
@lemmatized  вид                0.011582
             птица              0.011062
             территория         0.006171
             район              0.005858
             река               0.005784
                                  ...   
             минималистичный    0.000000
             натурщица          0.000000
             афрасиябнуть       0.000000
             nadh               0.000000
             рлэ                0.000000
Name: topic_18, Length: 61688, dtype: float32

In [61]:
model.class_ids

{'@lemmatized': 1}

In [62]:
KNOWN_METRICS

['euclidean', 'jensenshannon', 'hellinger', 'cosine']

In [30]:
MAIN_MODALITY

'@text'

In [29]:
def fit_and_compute_scores(model, dataset):
    model._fit(dataset.get_batch_vectorizer(), num_iterations=NUM_ITERATIONS)

    score_values = {
        'perplexity': model.scores[f'PerplexityScore{MAIN_MODALITY}'][-1],
    }

    phi = model.get_phi()

    # Currently all topics are taken into account
    target_topic_indices = list(range(NUM_TOPICS))  # phi.columns.get_loc()
    target_topic_names = [phi.columns[i] for i in target_topic_indices]

    
    top = NUM_TOP_TOKENS
    coherence_score = TopTokenCoherence(
        name=f'coherence_{top}',
        func=create_pmi_top_function(
            occurences, co_occurences,
            dataset.get_dataset().shape[0], [top],
            topic_indices=target_topic_indices,
            co_occurrences_smooth=1e-2,
        )
    )

    # print(f'Computing "{coherence_score._name}"...')
    
    value = coherence_score.call(model)
    score_values[coherence_score._name] = value
    topic_coherences = coherence_score.call_by_topic(model)
    topic_coherences = {t: float(v) for t, v in topic_coherences.items()}

    # print(f'Result by topic: {topic_coherences}.')


    # intra1 = IntratextCoherenceScore(
    #     name='toplen_pwt',
    #     data=dataset,
    #     computation_method=ComputationMethod.SEGMENT_LENGTH,
    #     word_topic_relatedness=WordTopicRelatednessType.PWT,
    #     should_compute=False,  # only on last iter
    # )
    intra2 = IntratextCoherenceScore(
        name='toplen_ptw',
        data=dataset,
        computation_method=ComputationMethod.SEGMENT_LENGTH,
        word_topic_relatedness=WordTopicRelatednessType.PTW,
        should_compute=False,
    )
    # intra3 = IntratextCoherenceScore(
    #     name='topden_ptw',
    #     data=dataset,
    #     computation_method=ComputationMethod.SUM_OVER_WINDOW,
    #     word_topic_relatedness=WordTopicRelatednessType.PTW,
    #     should_compute=False,
    # )
    # intra3_w4 = IntratextCoherenceScore(
    #     name='topden_ptw',
    #     data=dataset,
    #     computation_method=ComputationMethod.SUM_OVER_WINDOW,
    #     word_topic_relatedness=WordTopicRelatednessType.PTW,
    #     window=4,
    #     should_compute=False,
    # )

    intra_topic_coherences = dict()

    for intra in [intra2]:  # [intra2, intra3_w4]:  #[intra1, intra2, intra3]:
        # print(f'\nComputing "{intra._name}"...')

        current_intra_topic_coherences = intra.compute(model)

        assert all(v is not None for v in current_intra_topic_coherences.values())

        _values = current_intra_topic_coherences.values()

        current_intra_topic_coherences = {
            i: current_intra_topic_coherences[t]  # if v is not None else 0.0
            for i, t in enumerate(target_topic_names)
        }

        assert all(abs(x - y) <= 1e-6 for x, y in zip(_values, current_intra_topic_coherences.values())), (_values, current_intra_topic_coherences.values())  # "sorted" Python dicts
        
        intra_topic_coherences[f'topic_coherences_{intra._name}'] = current_intra_topic_coherences

        value = float(np.median(list(current_intra_topic_coherences.values())))
        score_values[intra._name] = value

        # print(f'Result by topic: {current_intra_topic_coherences}.')


    diversity_scores = [
        DiversityScore(
            name=f'diversity_{metric}',
            metric=metric,
            topic_names=target_topic_names,
            class_ids=MAIN_MODALITY,
        )
    
        for metric in KNOWN_METRICS
    ]
    
    for score in diversity_scores:
        value = score.call(model)
        score_values[score._name] = value

    
    return {
        'scores': score_values,
        'topic_coherences': topic_coherences,
        **intra_topic_coherences,
    }

In [30]:
co_occurences.shape

(22608, 22608)

In [39]:
BEST_PARAMS = dict()

## PLSA

In [59]:
PARAMS_EXPLORED[KnownModel.PLSA]

{}

In [60]:
NUM_TOPICS

20

In [61]:
results = []

for seed in range(NUM_TRAINS):
    print(seed)
    
    model = init_model_from_family(
        family=KnownModel.PLSA,
        dataset=dataset,
        main_modality=MAIN_MODALITY,
        num_topics=NUM_TOPICS,
        seed=seed,
    )
    scores = fit_and_compute_scores(model, dataset)
    results.append(scores)

0


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


In [40]:
BEST_PARAMS[KnownModel.PLSA] = None

In [41]:
dataset._data.shape

(2036, 3)

## Sparse

In [64]:
PARAMS_EXPLORED[KnownModel.SPARSE]

{'smooth_bcg_tau': [0.05, 0.1], 'sparse_sp_tau': [-0.05, -0.1]}

In [65]:
results = dict()

for sparse_sp_tau in PARAMS_EXPLORED[KnownModel.SPARSE]['sparse_sp_tau']:
    for smooth_bcg_tau in PARAMS_EXPLORED[KnownModel.SPARSE]['smooth_bcg_tau']:
        key = (sparse_sp_tau, smooth_bcg_tau)
        results[key] = []

        print(key)

        for seed in range(NUM_TRAINS):
            print(seed)
            
            model = init_model_from_family(
                family=KnownModel.SPARSE,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                model_params={'sparse_sp_tau': sparse_sp_tau, 'smooth_bcg_tau': smooth_bcg_tau}
            )

            for reg in model.regularizers.data:
                print(f"{reg}: {model.regularizers[reg].tau}")
 
            scores = fit_and_compute_scores(model, dataset)
            results[key].append(scores)

        print()

    print()

(-0.05, 0.05)
0


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 1.9718590531530529
smooth_theta_bcg: 21.895770861338022
sparse_phi_sp: -0.08920314764263809
sparse_theta_sp: -0.9905229675367199
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 1.9718590531530529
smooth_theta_bcg: 21.895770861338022
sparse_phi_sp: -0.08920314764263809
sparse_theta_sp: -0.9905229675367199
2


...

smooth_phi_bcg: 4.162813556656444
smooth_theta_bcg: 46.224405151713604
sparse_phi_sp: -0.17029691822685453
sparse_theta_sp: -1.8909983925701015
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 4.162813556656444
smooth_theta_bcg: 46.224405151713604
sparse_phi_sp: -0.17029691822685453
sparse_theta_sp: -1.8909983925701015
2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 4.162813556656444
smooth_theta_bcg: 46.224405151713604
sparse_phi_sp: -0.17029691822685453
sparse_theta_sp: -1.8909983925701015




In [66]:
results

{(-0.05,
  0.05): [{'scores': {'perplexity': 2465.832275390625,
    'coherence_20': array([0.65723241]),
    'diversity_euclidean': 0.06269599207736247,
    'diversity_jensenshannon': 0.6402356058322131,
    'diversity_hellinger': 0.7457255319298899,
    'diversity_cosine': 0.7392224332902931},
   'topic_coherences': {0: 0.4706941705736568,
    1: 0.9428906252054198,
    2: 0.570444932146175,
    3: 0.730668002345812,
    4: 0.5060028501686357,
    5: 0.4089519532813123,
    6: 0.91137477735875,
    7: 0.618962659949188,
    8: 0.5312373596697549,
    9: 0.6546315002705506,
    10: 0.8032429711947393,
    11: 0.5274397261902024,
    12: 0.8307535413071001,
    13: 0.6573898814990363,
    14: 0.46222355277065214,
    15: 0.6544428741574582,
    16: 0.9044616779054204,
    17: 0.6078315922424556,
    18: 0.6808253694723042,
    19: 0.6701780840189774}}, {'scores': {'perplexity': 2465.54345703125,
    'coherence_20': array([0.62574678]),
    'diversity_euclidean': 0.06272248450068274,
   

In [68]:
for k, r in results.items():
    mean_ppl = sum(v['scores']['perplexity'] for v in r) / len(r)
    print(k, mean_ppl)

(-0.05, 0.05) 2454.600341796875
(-0.05, 0.1) 2577.962646484375
(-0.1, 0.05) 2569.99609375
(-0.1, 0.1) 2695.2994791666665


In [69]:
# Best: (-0.05, 0.05) 2454.600341796875

In [42]:
BEST_PARAMS[KnownModel.SPARSE] = {
    'sparse_sp_tau': -0.05,
    'smooth_bcg_tau': 0.05,
}

In [71]:
model.get_phi()['topic_15'].sort_values(ascending=False)

modality  token       
@text     расстроиство    0.041009
          личность        0.013708
          шизофрения      0.012495
          человек         0.011630
          психический     0.008991
                            ...   
          наслаиваться    0.000000
          скелетный       0.000000
          валик           0.000000
          угольный        0.000000
          легально        0.000000
Name: topic_15, Length: 22608, dtype: float32

## Decorrelation

In [72]:
PARAMS_EXPLORED[KnownModel.DECORRELATION]

{'decorrelation_tau': [0.02, 0.05, 0.1]}

In [73]:
PARAMS_EXPLORED[KnownModel.ARTM]

{'smooth_bcg_tau': [0.05, 0.1],
 'sparse_sp_tau': [-0.05, -0.1],
 'decorrelation_tau': [0.02, 0.05, 0.1]}

In [74]:
DECORRELATION_TAUS = [0.01] + PARAMS_EXPLORED[KnownModel.DECORRELATION]['decorrelation_tau']

In [75]:
results = dict()

for decorrelation_tau in DECORRELATION_TAUS:
    for smooth_bcg_tau in PARAMS_EXPLORED[KnownModel.SPARSE]['smooth_bcg_tau']:
        key = (decorrelation_tau, smooth_bcg_tau)
        results[key] = []

        print(key)

        for seed in range(NUM_TRAINS):
            print(seed)
            
            model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                model_params={
                    'decorrelation_tau': decorrelation_tau,
                    'smooth_bcg_tau': smooth_bcg_tau,
                    'sparse_sp_tau': 0.0,
                }
            )

            for reg in model.regularizers.data:
                print(f"{reg}: {model.regularizers[reg].tau}")

            scores = fit_and_compute_scores(model, dataset)
            results[key].append(scores)

        print()

    print()

(0.01, 0.05)
0


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 1.9718590531530529
smooth_theta_bcg: 21.895770861338022
sparse_phi_sp: 0.0
sparse_theta_sp: 0.0
decorrelation: 0.01
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 1.9718590531530529
smooth_theta_bcg: 21.895770861338022
sparse_phi_sp: 0.0
sparse_theta_sp: 0.0
decorrelation: 0.01
2


...

smooth_phi_bcg: 4.162813556656444
smooth_theta_bcg: 46.224405151713604
sparse_phi_sp: 0.0
sparse_theta_sp: 0.0
decorrelation: 0.1
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 4.162813556656444
smooth_theta_bcg: 46.224405151713604
sparse_phi_sp: 0.0
sparse_theta_sp: 0.0
decorrelation: 0.1
2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 4.162813556656444
smooth_theta_bcg: 46.224405151713604
sparse_phi_sp: 0.0
sparse_theta_sp: 0.0
decorrelation: 0.1




In [79]:
len(results)

8

In [80]:
results

{(0.01,
  0.05): [{'scores': {'perplexity': 2269.100830078125,
    'coherence_20': array([0.6447714]),
    'diversity_euclidean': 0.05453546771942422,
    'diversity_jensenshannon': 0.6065282447749323,
    'diversity_hellinger': 0.6995732124401457,
    'diversity_cosine': 0.7049150689406795},
   'topic_coherences': {0: 0.4938616070784748,
    1: 0.8371804775325153,
    2: 0.4438504796964468,
    3: 0.7285835210890776,
    4: 0.5078567786356974,
    5: 0.4541532351151352,
    6: 0.8310553439006045,
    7: 0.6032429603377425,
    8: 0.808188785425736,
    9: 0.5908592555975367,
    10: 0.8160658668511684,
    11: 0.5530545787668294,
    12: 0.8281908937052933,
    13: 0.5575985763907658,
    14: 0.4901710044759395,
    15: 0.6330720539594581,
    16: 0.9980639632604971,
    17: 0.6184429737039088,
    18: 0.6046728654308625,
    19: 0.4972627448804702}}, {'scores': {'perplexity': 2263.556396484375,
    'coherence_20': array([0.64126859]),
    'diversity_euclidean': 0.056831659778020335,


In [81]:
for k, r in results.items():
    mean_ppl = sum(v['scores']['perplexity'] for v in r) / len(r)

    print(k, mean_ppl)

(0.01, 0.05) 2257.5345052083335
(0.01, 0.1) 2361.4169921875
(0.02, 0.05) 2257.9066569010415
(0.02, 0.1) 2361.5891927083335
(0.05, 0.05) 2269.4532063802085
(0.05, 0.1) 2372.4910481770835
(0.1, 0.05) 2345.3286946614585
(0.1, 0.1) 2440.0069986979165


In [82]:
#  Best:               (0.01, 0.05) 2257.5345052083335
# Close (very close): (0.02, 0.05) 2257.9066569010415

In [43]:
BEST_PARAMS[KnownModel.DECORRELATION] = {
    'decorrelation_tau': 0.02,
    'smooth_bcg_tau': 0.05,
}

## ARTM

In [84]:
PARAMS_EXPLORED[KnownModel.SPARSE]

{'smooth_bcg_tau': [0.05, 0.1], 'sparse_sp_tau': [-0.05, -0.1]}

In [85]:
PARAMS_EXPLORED[KnownModel.DECORRELATION]

{'decorrelation_tau': [0.02, 0.05, 0.1]}

In [86]:
PARAMS_EXPLORED[KnownModel.ARTM]

{'smooth_bcg_tau': [0.05, 0.1],
 'sparse_sp_tau': [-0.05, -0.1],
 'decorrelation_tau': [0.02, 0.05, 0.1]}

In [87]:
results = dict()

for decorrelation_tau in DECORRELATION_TAUS:
    for sparse_sp_tau in PARAMS_EXPLORED[KnownModel.ARTM]['sparse_sp_tau']:
        for smooth_bcg_tau in PARAMS_EXPLORED[KnownModel.SPARSE]['smooth_bcg_tau']:
            key = (decorrelation_tau, sparse_sp_tau, smooth_bcg_tau)
            results[key] = []
    
            print(key)
    
            for seed in range(NUM_TRAINS):
                print(seed)
                
                model = init_model_from_family(
                    family=KnownModel.ARTM,
                    dataset=dataset,
                    main_modality=MAIN_MODALITY,
                    num_topics=NUM_TOPICS,
                    seed=seed,
                    model_params={
                        'decorrelation_tau': decorrelation_tau,
                        'smooth_bcg_tau': smooth_bcg_tau,
                        'sparse_sp_tau': sparse_sp_tau,
                    }
                )
    
                for reg in model.regularizers.data:
                    print(f"{reg}: {model.regularizers[reg].tau}")
    
                scores = fit_and_compute_scores(model, dataset)
                results[key].append(scores)

            print()

        print()

    print()

(0.01, -0.05, 0.05)
0


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 1.9718590531530529
smooth_theta_bcg: 21.895770861338022
sparse_phi_sp: -0.08920314764263809
sparse_theta_sp: -0.9905229675367199
decorrelation: 0.01
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 1.9718590531530529
smooth_theta_bcg: 21.895770861338022
sparse_phi_sp: -0.08920314764263809
sparse_theta_sp: -0.9905229675367199
decorrelation: 0.01
2


...

smooth_phi_bcg: 4.162813556656444
smooth_theta_bcg: 46.224405151713604
sparse_phi_sp: -0.17029691822685453
sparse_theta_sp: -1.8909983925701015
decorrelation: 0.1
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 4.162813556656444
smooth_theta_bcg: 46.224405151713604
sparse_phi_sp: -0.17029691822685453
sparse_theta_sp: -1.8909983925701015
decorrelation: 0.1
2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 4.162813556656444
smooth_theta_bcg: 46.224405151713604
sparse_phi_sp: -0.17029691822685453
sparse_theta_sp: -1.8909983925701015
decorrelation: 0.1





In [91]:
len(results)

16

In [92]:
ppls = []

for k, r in results.items():
    mean_ppl = sum(v['scores']['perplexity'] for v in r) / len(r)
    ppls.append(mean_ppl)

    print(k, mean_ppl)

(0.01, -0.05, 0.05) 2458.5643717447915
(0.01, -0.05, 0.1) 2581.1787109375
(0.01, -0.1, 0.05) 2574.4801432291665
(0.01, -0.1, 0.1) 2698.272705078125
(0.02, -0.05, 0.05) 2464.08544921875
(0.02, -0.05, 0.1) 2585.5170084635415
(0.02, -0.1, 0.05) 2579.8678385416665
(0.02, -0.1, 0.1) 2702.2215983072915
(0.05, -0.05, 0.05) 2488.5302734375
(0.05, -0.05, 0.1) 2603.748291015625
(0.05, -0.1, 0.05) 2600.926025390625
(0.05, -0.1, 0.1) 2718.6932779947915
(0.1, -0.05, 0.05) 2552.7762858072915
(0.1, -0.05, 0.1) 2658.8765462239585
(0.1, -0.1, 0.05) 2662.7177734375
(0.1, -0.1, 0.1) 2775.323974609375


In [93]:
sorted(ppls)[:5]

[2458.5643717447915,
 2464.08544921875,
 2488.5302734375,
 2552.7762858072915,
 2574.4801432291665]

In [ ]:
#  Best: (0.01, -0.05, 0.05) 2458.5643717447915
# Close: (0.02, -0.05, 0.05) 2464.08544921875

In [44]:
BEST_PARAMS[KnownModel.DECORRELATION] = {
    'decorrelation_tau': 0.01,
    'sparse_sp_tau':    -0.05,
    'smooth_bcg_tau':    0.05,
}

## TLESS

In [45]:
PARAMS_EXPLORED[KnownModel.TLESS]

{}

In [96]:
results = []

for seed in range(NUM_TRAINS):
    print(seed)
    
    model = init_model_from_family(
        family=KnownModel.TLESS,
        dataset=dataset,
        main_modality=MAIN_MODALITY,
        num_topics=NUM_TOPICS,
        seed=seed,
    )
    scores = fit_and_compute_scores(model, dataset)
    results.append(scores)

0


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


In [97]:
results

[{'scores': {'perplexity': 2710.05322265625,
   'coherence_20': array([0.69064164]),
   'diversity_euclidean': 0.10360478401886986,
   'diversity_jensenshannon': 0.7566075022840391,
   'diversity_hellinger': 0.8854519637041928,
   'diversity_cosine': 0.9298865715901761},
  'topic_coherences': {0: 0.4369066763829261,
   1: 1.3688887093860056,
   2: 0.5734527878692386,
   3: 0.555006268470112,
   4: 0.5031285424754262,
   5: 0.5165577102590853,
   6: 0.917272974090113,
   7: 0.8170612891678282,
   8: 0.6308371342605484,
   9: 0.6249737112288283,
   10: 0.7587532979791912,
   11: 0.9109564791512955,
   12: 0.7263835755690161,
   13: 0.6870362602850006,
   14: 0.35765613593653406,
   15: 0.6732874851412556,
   16: 0.8520000816925969,
   17: 0.4636096696813935,
   18: 1.0211192239453564,
   19: 0.41794472443348246}},
 {'scores': {'perplexity': 2713.791259765625,
   'coherence_20': array([0.66324343]),
   'diversity_euclidean': 0.10462702946308376,
   'diversity_jensenshannon': 0.75628969734

In [98]:
# Best:

In [46]:
BEST_PARAMS[KnownModel.TLESS] = None

## LDA

In [47]:
PARAMS_EXPLORED[KnownModel.LDA]

{'prior': ['symmetric', 'asymmetric', 'heuristic']}

In [101]:
results = dict()

for prior in PARAMS_EXPLORED[KnownModel.LDA]['prior']:
    key = prior
    results[key] = []

    print(key)

    for seed in range(NUM_TRAINS):
        print(seed)
        
        model = init_model_from_family(
            family=KnownModel.LDA,
            dataset=dataset,
            main_modality=MAIN_MODALITY,
            num_topics=NUM_TOPICS,
            seed=seed,
            model_params={'prior': prior}
        )

        for reg in model.regularizers.data:
            print(f"{reg}: {model.regularizers[reg].tau}")

        scores = fit_and_compute_scores(model, dataset)
        results[key].append(scores)

    print()

symmetric
0


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi: 0.05
smooth_theta: 0.05
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi: 0.05
smooth_theta: 0.05
2


...

smooth_phi: 0.01
smooth_theta: 2.5
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi: 0.01
smooth_theta: 2.5
2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi: 0.01
smooth_theta: 2.5



In [105]:
results

{'symmetric': [{'scores': {'perplexity': 2206.36328125,
    'coherence_20': array([0.58326465]),
    'diversity_euclidean': 0.04829135840773115,
    'diversity_jensenshannon': 0.5641920329897929,
    'diversity_hellinger': 0.6244635499992061,
    'diversity_cosine': 0.6530752667822146},
   'topic_coherences': {0: 0.41603726501551547,
    1: 0.7613769777112703,
    2: 0.4178029383666581,
    3: 0.649953932720543,
    4: 0.4980276984955354,
    5: 0.48879998830907045,
    6: 0.7867876162181322,
    7: 0.49511778383633126,
    8: 0.705775085791701,
    9: 0.5609756518683418,
    10: 0.7079941341971158,
    11: 0.5274557253976083,
    12: 0.7109965752522808,
    13: 0.5510059357376327,
    14: 0.3947668738392959,
    15: 0.46749608493419537,
    16: 0.7961542891063786,
    17: 0.6018775126751085,
    18: 0.6035551172078581,
    19: 0.5233358795514585}},
  {'scores': {'perplexity': 2201.345703125,
    'coherence_20': array([0.58114333]),
    'diversity_euclidean': 0.05324276936292553,
    '

In [106]:
ppls = []

for k, r in results.items():
    mean_ppl = sum(v['scores']['perplexity'] for v in r) / len(r)
    ppls.append(mean_ppl)

    print(k, mean_ppl)

symmetric 2195.3936360677085
asymmetric 2195.6841634114585
heuristic 2337.3949381510415


In [104]:
sorted(ppls)

[2195.3936360677085, 2195.6841634114585, 2337.3949381510415]

In [94]:
# Best:               asymmetric 2195.6841634114585
# Close (very close): symmetric 2195.3936360677085

In [48]:
BEST_PARAMS[KnownModel.LDA] = {
    'prior': 'symmetric',
}

In [108]:
BEST_PARAMS

{<KnownModel.PLSA: 'PLSA'>: None,
 <KnownModel.SPARSE: 'sparse'>: {'sparse_sp_tau': -0.05,
  'smooth_bcg_tau': 0.05},
 <KnownModel.DECORRELATION: 'decorrelation'>: {'decorrelation_tau': 0.01,
  'sparse_sp_tau': -0.05,
  'smooth_bcg_tau': 0.05},
 <KnownModel.TLESS: 'TARTM'>: None,
 <KnownModel.LDA: 'LDA'>: {'prior': 'symmetric'}}

In [49]:
BEST_PARAMS

{<KnownModel.PLSA: 'PLSA'>: None,
 <KnownModel.SPARSE: 'sparse'>: {'sparse_sp_tau': -0.05,
  'smooth_bcg_tau': 0.05},
 <KnownModel.DECORRELATION: 'decorrelation'>: {'decorrelation_tau': 0.01,
  'sparse_sp_tau': -0.05,
  'smooth_bcg_tau': 0.05},
 <KnownModel.TLESS: 'TARTM'>: None,
 <KnownModel.LDA: 'LDA'>: {'prior': 'symmetric'}}

In [31]:
BEST_PARAMS = {KnownModel.PLSA: None,
 KnownModel.SPARSE: {'sparse_sp_tau': -0.05,
  'smooth_bcg_tau': 0.05},
 KnownModel.DECORRELATION: {'decorrelation_tau': 0.01,
  'sparse_sp_tau': -0.05,
  'smooth_bcg_tau': 0.05},
 KnownModel.TLESS: None,
 KnownModel.LDA: {'prior': 'symmetric'}}

In [32]:
import json
import warnings

warnings.simplefilter('ignore', UserWarning)

In [54]:
NUM_TRAINS = 20  # 100
COHERENCES = {
    'topic_coherences': list(),
    # 'topic_coherences_toplen_pwt': list(),
    'topic_coherences_toplen_ptw': list(),
    # 'topic_coherences_topden_ptw': list(),
}

In [34]:
! ls results50_intra

mkb10  postnauka


In [38]:
! ls results50_intra/mkb10

decorrelation_with_cohs.json  plsa_with_cohs.json    tless_with_cohs.json
lda_with_cohs.json	      sparse_with_cohs.json


In [39]:
! cat results50_intra/mkb10/lda_with_cohs.json

[
    {
        "scores": {
            "perplexity": 1659.3775634765625,
            "coherence_20": 0.7003232183080772,
            "toplen_ptw": 1.4195736891509028,
            "diversity_euclidean": 0.063462280204253,
            "diversity_jensenshannon": 0.6049740930647077,
            "diversity_hellinger": 0.6778949012201013,
            "diversity_cosine": 0.7560758241499191
        },
        "topic_coherences": {
            "0": 0.5983527246499233,
            "1": 0.8776372492610116,
            "2": 0.49963712528959275,
            "3": 0.9047741192144153,
            "4": 0.5765406927744863,
            "5": 0.35141993939439753,
            "6": 0.738815977401959,
            "7": 0.7784221108894375,
            "8": 0.6262989498000313,
            "9": 0.6264729370416072,
            "10": 0.6458553325813213,
            "11": 0.5499689353985209,
            "12": 0.5683727972880226,
            "13": 0.7310737841309272,
            "14": 0.5049207892688458,
           

In [40]:
import json

In [42]:
d1 = json.loads(open('results50_intra/mkb10/decorrelation_with_cohs.json').read())

In [43]:
d2 = json.loads(open('results50_intra/mkb10/lda_with_cohs.json').read())

In [45]:
len(d2)

20

In [46]:
SAVE_FOLDER = 'results50_intra/mkb10'

! mkdir -p $SAVE_FOLDER

In [53]:
! ls

20_Newsgroups__internals
ARTM-Models-20NewsGroups-T20.ipynb
ARTM-Models-20NewsGroups-T50.ipynb
ARTM-Models-MKB10-T20-Copy1.ipynb
ARTM-Models-MKB10-T20.ipynb
ARTM-Models-MKB10-T50.ipynb
ARTM-Models-PostNauka-T20.ipynb
ARTM-Models-PostNauka-T50.ipynb
ARTM-Models-RTL-Wiki-Person-T20.ipynb
ARTM-Models-RTL-Wiki-Person-T50.ipynb
ARTM-Models-RuWikiGood-T20.ipynb
ARTM-Models-RuWikiGood-T50.ipynb
BERTopic
Experiment_v2.ipynb
Experiment_v3.ipynb
Good_RU_Wiki__internals
Iterative-Model-20NewsGroups-T20.ipynb
Iterative-Model-MKB10-T20.ipynb
Iterative-Model-PostNauka-T20.ipynb
Iterative-Model-PostNauka-T50.ipynb
Iterative-Model-RTL-Wiki-Person-T20.ipynb
Iterative-Model-RuWikiGood-T20.ipynb
MKB_10__internals
phi1.batch
Post_Science__internals
results
results50
RTL_Wiki_Person__internals
test_dict.txt
TestTopicNet.ipynb
TopicBank-20NewsGroups-T20.ipynb
TopicBank-MKB10-T20.ipynb
TopicBank-PostNauka-T20-Copy1.ipynb
TopicBank-PostNauka-T20.ipynb
TopicBank-RTL-Wiki-Person-T20.ipynb
TopicBank-RuWikiGood-T

In [55]:
for file_name in """decorrelation_with_cohs.json  plsa_with_cohs.json    tless_with_cohs.json
lda_with_cohs.json	      sparse_with_cohs.json""".split():
    d = json.loads(open(f'results50_intra/mkb10/{file_name}').read())

    for m in d:
        for k in COHERENCES:
            COHERENCES[k].extend(list(m[k].values()))

In [56]:
for k in COHERENCES:
    print(k)
    print(len(COHERENCES[k]))

topic_coherences
5000
topic_coherences_toplen_ptw
5000


In [53]:
m

{'scores': {'perplexity': 1872.85546875,
  'coherence_20': 0.7493941020417191,
  'toplen_ptw': 1.3972322466329645,
  'diversity_euclidean': 0.07805532729702504,
  'diversity_jensenshannon': 0.662331031457289,
  'diversity_hellinger': 0.7745897771075634,
  'diversity_cosine': 0.8060450757863635},
 'topic_coherences': {'0': 0.668558936598835,
  '1': 0.8972493434787214,
  '2': 0.8011708418868666,
  '3': 0.859702049143148,
  '4': 0.5578829646493697,
  '5': 1.1919245433150552,
  '6': 0.4369665719644201,
  '7': 0.9604689913703685,
  '8': 0.8563398007387278,
  '9': 0.639201458202056,
  '10': 0.796127934050161,
  '11': 0.6725877512987641,
  '12': 0.5903678904077376,
  '13': 0.7966969899957382,
  '14': 0.7927836958978953,
  '15': 0.7017723464653246,
  '16': 0.5262504900352136,
  '17': 0.563367632795984,
  '18': 0.931331457766372,
  '19': 0.6246148770329477,
  '20': 0.6665276368180338,
  '21': 0.5653939188825168,
  '22': 0.8468959337712114,
  '23': 0.5717967780193528,
  '24': 1.1353010258830516,

In [35]:
# PLSA

start = time.time()

scores = []

# for seed in tqdm(range(NUM_TRAINS), total=NUM_TRAINS):
for seed in range(NUM_TRAINS):
    if seed != NUM_TRAINS - 1:
        print(seed, end=' ')
    else:
        print(seed)

    model = init_model_from_family(
        family=KnownModel.PLSA,
        dataset=dataset,
        main_modality=MAIN_MODALITY,
        num_topics=NUM_TOPICS,
        seed=seed,
    )
    results = fit_and_compute_scores(model, dataset)
    # scores.append(results['scores'])
    scores.append(results)

    # COHERENCES.extend(
    #     list(results['topic_coherences'].values())
    # )

    for k in COHERENCES:
        COHERENCES[k].extend(
            list(results[k].values())
        )

end = time.time()

print(f'Elapsed: {(end - start) / 60} min')

0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19
Elapsed: 62.80538218816121 min


In [39]:
scores[0]

{'scores': {'perplexity': 1638.628173828125,
  'coherence_20': array([0.68967517]),
  'diversity_euclidean': 0.06583711691451596,
  'diversity_jensenshannon': 0.6288936400736841,
  'diversity_hellinger': 0.7306780758379043,
  'diversity_cosine': 0.759849721283957},
 'topic_coherences': {0: 0.6479883527217851,
  1: 0.8776372492610116,
  2: 0.5620395670945291,
  3: 0.9163011462957861,
  4: 0.5765406927744862,
  5: 0.35141993939439753,
  6: 0.7206498735780156,
  7: 0.7887862302181597,
  8: 0.6262989498000314,
  9: 0.5773043988467672,
  10: 0.632171489274859,
  11: 0.49838752061888303,
  12: 0.6302083898568993,
  13: 0.7310737841309272,
  14: 0.5503903287793733,
  15: 0.7671210276301571,
  16: 1.0355785244038034,
  17: 0.6215492871402313,
  18: 0.6526065226404881,
  19: 0.9440447181157697,
  20: 0.9648672535659765,
  21: 0.5662728294453778,
  22: 0.5919412024847805,
  23: 0.9741333700021619,
  24: 0.7909556733824701,
  25: 0.8356659392060041,
  26: 0.5490337957277347,
  27: 0.7499151637642

In [36]:
# for s in scores:
#     s['coherence_20'] = float(s['coherence_20'])

for s in scores:
    s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

In [57]:
scores

[{'perplexity': 1638.628173828125,
  'coherence_20': 0.6896751669567669,
  'diversity_euclidean': 0.06583711264916696,
  'diversity_jensenshannon': 0.628893642445529,
  'diversity_hellinger': 0.7306780691288125,
  'diversity_cosine': 0.7598497153975247},
 {'perplexity': 1669.8839111328125,
  'coherence_20': 0.6736719237515266,
  'diversity_euclidean': 0.06377540516290432,
  'diversity_jensenshannon': 0.6311223780089505,
  'diversity_hellinger': 0.7330348843074705,
  'diversity_cosine': 0.7474155148798083},
 {'perplexity': 1670.3743896484375,
  'coherence_20': 0.6594187599725305,
  'diversity_euclidean': 0.06277147566314643,
  'diversity_jensenshannon': 0.627427741544272,
  'diversity_hellinger': 0.7286301355803733,
  'diversity_cosine': 0.7396285471085464},
 {'perplexity': 1646.3388671875,
  'coherence_20': 0.7087392007472575,
  'diversity_euclidean': 0.06584460623893432,
  'diversity_jensenshannon': 0.6284734706030459,
  'diversity_hellinger': 0.7300590079048894,
  'diversity_cosine':

In [58]:
SAVE_FOLDER

'results50/mkb10'

In [37]:
with open(SAVE_FOLDER + '/plsa_with_cohs.json', 'w') as f:
    f.write(
        json.dumps(scores, indent=4)
    )

In [60]:
len(COHERENCES)

1000

In [61]:
COHERENCES[:10]

[0.6479883527217851,
 0.8776372492610116,
 0.5620395670945291,
 0.9163011462957861,
 0.5765406927744862,
 0.35141993939439753,
 0.7206498735780156,
 0.7887862302181597,
 0.6262989498000314,
 0.5773043988467672]

In [62]:
COHERENCES[:20]

[0.6479883527217851,
 0.8776372492610116,
 0.5620395670945291,
 0.9163011462957861,
 0.5765406927744862,
 0.35141993939439753,
 0.7206498735780156,
 0.7887862302181597,
 0.6262989498000314,
 0.5773043988467672,
 0.632171489274859,
 0.49838752061888303,
 0.6302083898568993,
 0.7310737841309272,
 0.5503903287793733,
 0.7671210276301571,
 1.0355785244038034,
 0.6215492871402313,
 0.6526065226404881,
 0.9440447181157697]

In [38]:
# Sparse

start = time.time()

scores = []

# for seed in tqdm(range(NUM_TRAINS), total=NUM_TRAINS):
for seed in range(NUM_TRAINS):
    if seed != NUM_TRAINS - 1:
        print(seed, end=' ')
    else:
        print(seed)

    model = init_model_from_family(
        family=KnownModel.SPARSE,
        dataset=dataset,
        main_modality=MAIN_MODALITY,
        num_topics=NUM_TOPICS,
        seed=seed,
        model_params=BEST_PARAMS[KnownModel.SPARSE],
    )

    if seed == 0:
        for reg in model.regularizers.data:
            print(f"{reg}: {model.regularizers[reg].tau}")

    results = fit_and_compute_scores(model, dataset)
    # scores.append(results['scores'])
    scores.append(results)

    # COHERENCES.extend(
    #     list(results['topic_coherences'].values())
    # )

    for k in COHERENCES:
        COHERENCES[k].extend(
            list(results[k].values())
        )

end = time.time()

print(f'Elapsed: {(end - start) / 60} min')

0 smooth_phi_bcg: 1.9718590531530529
smooth_theta_bcg: 21.895770861338022
sparse_phi_sp: -0.035681259057055235
sparse_theta_sp: -0.396209187014688
1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19
Elapsed: 65.06854359308879 min


In [39]:
# for s in scores:
#     s['coherence_20'] = float(s['coherence_20'])

for s in scores:
    s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

In [40]:
with open(SAVE_FOLDER + '/sparse_with_cohs.json', 'w') as f:
    f.write(
        json.dumps(scores, indent=4)
    )

In [66]:
len(COHERENCES)

2000

In [67]:
COHERENCES[-10:]

[0.6498668235160833,
 1.0036003644846851,
 0.6079990829041929,
 0.7293497869873817,
 0.9067311210073428,
 0.8769690432596147,
 0.9530955345494656,
 0.7429337101599988,
 0.8812153606423785,
 0.7855946767737623]

In [68]:
max(COHERENCES)

1.6000897691737095

In [69]:
min(COHERENCES)

0.3219850241241305

In [41]:
def train_many(model_family, save_file_path):
    start = time.time()
    
    scores = []

    # for seed in tqdm(range(NUM_TRAINS), total=NUM_TRAINS):
    for seed in range(NUM_TRAINS):
        if seed != NUM_TRAINS - 1:
            print(seed, end=' ')
        else:
            print(seed)
    
        model = init_model_from_family(
            family=model_family,
            dataset=dataset,
            main_modality=MAIN_MODALITY,
            num_topics=NUM_TOPICS,
            seed=seed,
            model_params=BEST_PARAMS[model_family],
        )
    
        if seed == 0:
            for reg in model.regularizers.data:
                print(f"{reg}: {model.regularizers[reg].tau}")
    
        results = fit_and_compute_scores(model, dataset)
        # scores.append(results['scores'])
        scores.append(results)
    
        # COHERENCES.extend(
        #     list(results['topic_coherences'].values())
        # )
    
        for k in COHERENCES:
            COHERENCES[k].extend(
                list(results[k].values())
            )
    
    end = time.time()
    
    print(f'Elapsed: {(end - start) / 60} min')

    # for s in scores:
    #     s['coherence_20'] = float(s['coherence_20'])
    
    for s in scores:
        s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

    with open(save_file_path, 'w') as f:
        f.write(
            json.dumps(scores, indent=4)
        )

In [42]:
train_many(KnownModel.DECORRELATION, SAVE_FOLDER + '/decorrelation_with_cohs.json')

0 decorrelation: 0.01
smooth_phi_bcg: 1.9718590531530529
smooth_theta_bcg: 21.895770861338022
1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19
Elapsed: 64.05706568956376 min


In [47]:
len(COHERENCES)

3000

In [43]:
train_many(KnownModel.TLESS, SAVE_FOLDER + '/tless_with_cohs.json')

0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19
Elapsed: 64.48921434481939 min


In [49]:
len(COHERENCES)

4000

In [ ]:
train_many(KnownModel.LDA, SAVE_FOLDER + '/lda_with_cohs.json')

0 smooth_phi: 0.02
smooth_theta: 0.02
1 2 3 4 5 6 

In [52]:
len(COHERENCES)

5000

In [77]:
COHERENCES

[0.6479883527217851,
 0.8776372492610116,
 0.5620395670945291,
 0.9163011462957861,
 0.5765406927744862,
 0.35141993939439753,
 0.7206498735780156,
 0.7887862302181597,
 0.6262989498000314,
 0.5773043988467672,
 0.632171489274859,
 0.49838752061888303,
 0.6302083898568993,
 0.7310737841309272,
 0.5503903287793733,
 0.7671210276301571,
 1.0355785244038034,
 0.6215492871402313,
 0.6526065226404881,
 0.9440447181157697,
 0.9648672535659765,
 0.5662728294453778,
 0.5919412024847805,
 0.9741333700021619,
 0.7909556733824701,
 0.8356659392060041,
 0.5490337957277347,
 0.7499151637642794,
 0.49882480457036715,
 1.0093733590209746,
 0.7380210914068371,
 0.6232122189510844,
 0.6170087310336526,
 0.48610972011770753,
 0.762523457862982,
 0.7136096082509353,
 0.555931122531894,
 0.7605876909166162,
 0.6082890287430017,
 0.6403930618386158,
 1.0084299264741095,
 0.5466840002831019,
 0.6469580365284837,
 0.5014913015670767,
 0.4604676067662721,
 0.5893851134803278,
 1.0622174091248615,
 0.589284025

In [1]:
1

1

In [57]:
COHERENCES.keys()

dict_keys(['topic_coherences', 'topic_coherences_toplen_ptw'])

In [58]:
for k in COHERENCES:
    print(k)
    print(len(COHERENCES[k]))

topic_coherences
5000
topic_coherences_toplen_ptw
5000


In [86]:
for p in range(5, 100, 5):
    print(f'{p:2}: {np.percentile(COHERENCES, p)}')

 5: 0.4437824489096678
10: 0.48828982029060863
15: 0.5234896551330657
20: 0.5504721115444576
25: 0.5742801728246323
30: 0.5994824270016577
35: 0.6205541977162965
40: 0.6436919251600118
45: 0.668827385009165
50: 0.6932423733170683
55: 0.7199442529469555
60: 0.749189874927616
65: 0.7786031543833574
70: 0.8093978586212078
75: 0.8489776360689408
80: 0.8997333820973618
85: 0.9659319151034702
90: 1.0544318710389238
95: 1.1927589824669111


In [60]:
for k in COHERENCES:
    print(k)

    for p in range(5, 100, 5):
        print(f'{p:2}: {np.percentile(COHERENCES[k], p)}')

    print()

topic_coherences
 5: 0.4437824489096678
10: 0.48828982029060863
15: 0.5233770099388192
20: 0.5504721115444576
25: 0.5742801728246323
30: 0.5994824270016577
35: 0.6205541977162965
40: 0.6436919251600118
45: 0.668827385009165
50: 0.6932423733170683
55: 0.7199442529469555
60: 0.749189874927616
65: 0.7786031543833574
70: 0.8093978586212078
75: 0.8489776360689408
80: 0.8997333820973618
85: 0.9659319151034702
90: 1.0544318710389238
95: 1.1927589824669111

topic_coherences_toplen_ptw
 5: 1.2244212404273518
10: 1.2499085596596853
15: 1.2721760746781052
20: 1.2910160325747209
25: 1.3087625393890354
30: 1.3278679394465835
35: 1.3439167332395403
40: 1.3621972408950418
45: 1.3808243188238345
50: 1.402003050699888
55: 1.4251673409938175
60: 1.4480698704586892
65: 1.4737430603550887
70: 1.5041606723306686
75: 1.5370951394122865
80: 1.5728590187443703
85: 1.6225986769479606
90: 1.6973119879327287
95: 1.8740279244432314



In [87]:
min(COHERENCES), max(COHERENCES)

(0.22000498229980117, 1.9214525545047356)

In [80]:
np.argmin(COHERENCES), np.argmax(COHERENCES)

(2549, 3285)

In [61]:
for k in COHERENCES:
    print(k)
    print(min(COHERENCES[k]), max(COHERENCES[k]))
    print()

topic_coherences
0.22000498229980117 1.9214525545047356

topic_coherences_toplen_ptw
1.0978916085866899 2.709184521312321



In [84]:
for p in [2, 98]:
    print(f'{p:2}: {np.percentile(COHERENCES, p)}')

 2: 0.3980679204076211
98: 1.3832254967588362


In [82]:
1

1

In [147]:
BEST_PARAMS

{<KnownModel.PLSA: 'PLSA'>: None,
 <KnownModel.SPARSE: 'sparse'>: {'sparse_sp_tau': -0.05,
  'smooth_bcg_tau': 0.05},
 <KnownModel.DECORRELATION: 'decorrelation'>: {'decorrelation_tau': 0.01,
  'sparse_sp_tau': -0.05,
  'smooth_bcg_tau': 0.05},
 <KnownModel.TLESS: 'TARTM'>: None,
 <KnownModel.LDA: 'LDA'>: {'prior': 'symmetric'}}

In [62]:
dataset.get_dictionary()

artm.Dictionary(name=849b3a43-4e71-459c-90df-177ca1afa29f, num_entries=22608)

In [138]:
COHERENCES[-11:]

[0.9151910589643388,
 0.40117780835167877,
 0.4405340306270155,
 0.4395601942762919,
 0.58209845522529,
 0.6616468035210405,
 0.4973913152802322,
 0.5207985518820669,
 0.6030229525188386,
 1.2961590770289348,
 0.6465084701742573]

In [63]:
COHERENCES

{'topic_coherences': [0.5661269332907423,
  0.991777197523978,
  0.5449996835453096,
  0.936561873069384,
  0.5502717192118143,
  0.48717920626091316,
  0.8170702362895506,
  0.8336404345838453,
  0.621791195674679,
  0.7384452225620302,
  0.6149971280317943,
  0.6019472307155681,
  0.8114435376424296,
  0.7950022207291308,
  0.5417127298312457,
  0.8800055898709738,
  1.0945885347717026,
  0.6729692193654769,
  0.911596165233027,
  0.7721563535686812,
  1.0141433666218365,
  0.6523445893523582,
  0.6558165424852667,
  1.0651099348988158,
  0.9199008242596308,
  0.9291757262977173,
  0.6356648720796632,
  0.8777004635360881,
  0.5327073197185466,
  1.1902429600454634,
  0.7971695888469837,
  0.6267749837617903,
  0.5919140296487906,
  0.5209025353876484,
  0.8104820065060407,
  0.8303795842438235,
  0.6373805962468747,
  0.8178394758765531,
  0.7360007140762466,
  0.6712060713350534,
  1.1827550164551444,
  0.5794427270209325,
  0.7265554741716828,
  0.610790797016425,
  0.557458426190

## Newman Vs. Intratext

### Correlation

In [64]:
from scipy.stats import spearmanr

In [65]:
spearmanr(
    COHERENCES['topic_coherences'],
    COHERENCES['topic_coherences_toplen_ptw'],
)

SignificanceResult(statistic=0.33552757607580025, pvalue=7.989782696210074e-132)

### Tops Intersection

In [66]:
inds1 = np.argsort(COHERENCES['topic_coherences'])[::-1][:100]

In [67]:
inds1 = set(np.argsort(COHERENCES['topic_coherences'])[::-1][:100])
inds2 = set(np.argsort(COHERENCES['topic_coherences_toplen_ptw'])[::-1][:100])

In [68]:
len(inds1 & inds2)

16

In [69]:
inds1 = set(np.argsort(COHERENCES['topic_coherences'])[::-1][:500])
inds2 = set(np.argsort(COHERENCES['topic_coherences_toplen_ptw'])[::-1][:500])

In [70]:
len(inds1 & inds2)

179

### Newman-in-Intratext Density (Mutual Density / Intra-Top Density)

In [71]:
inds2 = np.argsort(COHERENCES['topic_coherences_toplen_ptw'])[::-1][:100]

In [76]:
np.quantile(
    np.array(COHERENCES['topic_coherences']), 0.5
)

0.6932423733170683

In [72]:
np.mean(
    np.array(COHERENCES['topic_coherences'])
)

0.7376527521830177

In [73]:
np.mean(
    np.array(COHERENCES['topic_coherences'])[inds2]
)

0.7960414123648581

In [74]:
inds2 = np.argsort(COHERENCES['topic_coherences_toplen_ptw'])[::-1][:200]

In [75]:
np.mean(
    np.array(COHERENCES['topic_coherences'])[inds2]
)

0.932864241921701